# Notebook 1: Dataset Preparation

This notebook:
1. Loads and preprocesses the multiclass sentiment dataset from Hugging Face
2. Converts examples into instruction-style prompts
3. Tokenizes datasets for DistilGPT-2 (max length 128; pad_token=eos_token; labels=input_ids for causal LM)
4. Saves tokenized training/test splits to disk
5. Builds and saves a larger BoolQ evaluation set from `google/boolq` (default 500 examples)

In [1]:
# Import required libraries
import os
import json
import random
from datasets import load_dataset
from transformers import AutoTokenizer
import torch

# Set random seeds for reproducibility
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Load Sentiment Dataset

In [2]:
# Load the multiclass sentiment dataset from Hugging Face
dataset_name = "Sp1786/multiclass-sentiment-analysis-dataset"
print(f"Loading dataset: {dataset_name}")

dataset = load_dataset(dataset_name)
print(f"\nDataset keys: {dataset.keys()}")
print(f"Dataset structure: {dataset}")

# Check the label mapping
labels = {0: "negative", 1: "neutral", 2: "positive"}
print(f"\nLabel mapping: {labels}")

# Display a few examples
print("\nSample examples:")
for i in range(min(3, len(dataset['train']))):
    example = dataset['train'][i]
    print(f"\nExample {i+1}:")
    print(f"  Text: {example.get('text', example.get('sentence', 'N/A'))}")
    print(f"  Label: {example.get('label', example.get('sentiment', 'N/A'))} ({labels.get(example.get('label', example.get('sentiment', -1)), 'unknown')})")

Loading dataset: Sp1786/multiclass-sentiment-analysis-dataset

Dataset keys: dict_keys(['train', 'validation', 'test'])
Dataset structure: DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'sentiment'],
        num_rows: 31232
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'sentiment'],
        num_rows: 5205
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'sentiment'],
        num_rows: 5206
    })
})

Label mapping: {0: 'negative', 1: 'neutral', 2: 'positive'}

Sample examples:

Example 1:
  Text: Cooking microwave pizzas, yummy
  Label: 2 (positive)

Example 2:
  Text: Any plans of allowing sub tasks to show up in the widget?
  Label: 1 (neutral)

Example 3:
  Text:  I love the humor, I just reworded it. Like saying 'group therapy' instead`a 'gang banging'. Keeps my moms off my back.   Hahaha
  Label: 2 (positive)


## 2. Create Instruction-Style Prompts

In [3]:
def create_sentiment_prompt(example, labels):
    """
    Converts a sentiment example into an instruction-style prompt.
    
    Args:
        example: Dataset example with text and label
        labels: Dictionary mapping label IDs to label strings
        
    Returns:
        prompt: Instruction prompt string
        target: Label string (e.g., "negative", "neutral", "positive")
        text_for_lm: Full LM training string
    """
    # Handle different possible field names for text and label
    text = example.get('text', example.get('sentence', example.get('review', '')))
    label_id = example.get('label', example.get('sentiment', -1))
    
    # Get the label string
    target = labels.get(label_id, "unknown")
    
    # Create the instruction-style prompt
    prompt = f"Text: {text}\nQuestion: What is the sentiment? (negative, neutral, positive)\nAnswer:"
    
    # The LM training string combines prompt and target
    text_for_lm = prompt + " " + target
    
    return prompt, target, text_for_lm

# Test the function on a sample
print("Testing prompt creation:")
test_example = dataset['train'][0]
prompt, target, text_for_lm = create_sentiment_prompt(test_example, labels)
print(f"\nPrompt:\n{prompt}")
print(f"\nTarget: {target}")
print(f"\nFull LM training string:\n{text_for_lm}")

Testing prompt creation:

Prompt:
Text: Cooking microwave pizzas, yummy
Question: What is the sentiment? (negative, neutral, positive)
Answer:

Target: positive

Full LM training string:
Text: Cooking microwave pizzas, yummy
Question: What is the sentiment? (negative, neutral, positive)
Answer: positive


## 3. Tokenize Datasets for DistilGPT-2

In [4]:
# Load the DistilGPT-2 tokenizer
model_name = "distilgpt2"
print(f"Loading tokenizer for {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set padding token to eos_token to avoid warnings
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")

Loading tokenizer for distilgpt2
Tokenizer loaded. Vocab size: 50257
Pad token: <|endoftext|>
EOS token: <|endoftext|>


In [ ]:
def tokenize_dataset(examples, labels, tokenizer, max_length=128):
    """
    Tokenize a batch for causal LM:
    - Build instruction-style text_for_lm = prompt + " " + target
    - Truncate to max_length (128)
    - Use padding='max_length' with pad_token=eos_token
    - Set labels = input_ids (standard causal LM convention)
    """
    texts_for_lm = []
    
    for i in range(len(examples.get('label', examples.get('sentiment', [])))):
        label_id = examples.get('label', examples.get('sentiment', []))[i]
        text_fields = ['text', 'sentence', 'review']
        text = None
        for field in text_fields:
            if field in examples:
                text = examples[field][i]
                break
        if text is None:
            text = ""
        
        _, _, text_for_lm = create_sentiment_prompt({'text': text, 'label': label_id}, labels)
        texts_for_lm.append(text_for_lm)
    
    tokenized = tokenizer(
        texts_for_lm,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors=None
    )
    
    # Causal LM supervision: predict next token; targets are the same as inputs
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

# Apply tokenization to train and test splits
print("Tokenizing training set...")
tokenized_train = dataset['train'].map(
    lambda examples: tokenize_dataset(examples, labels, tokenizer, max_length=128),
    batched=True,
    remove_columns=dataset['train'].column_names
)

print("Tokenizing test set...")
tokenized_test = dataset['test'].map(
    lambda examples: tokenize_dataset(examples, labels, tokenizer, max_length=128),
    batched=True,
    remove_columns=dataset['test'].column_names
)

print(f"\nTokenized train set size: {len(tokenized_train)}")
print(f"Tokenized test set size: {len(tokenized_test)}")
print(f"Tokenized train features: {tokenized_train.features}")

Tokenizing training set...
Tokenizing test set...

Tokenized train set size: 31232
Tokenized test set size: 5206
Tokenized train features: {'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}


## 4. Save Tokenized Datasets to Disk

In [6]:
# Create data directory if it doesn't exist
os.makedirs("data/tokenized_sentiment_train", exist_ok=True)
os.makedirs("data/tokenized_sentiment_test", exist_ok=True)

# Save tokenized datasets
print("Saving tokenized training set...")
tokenized_train.save_to_disk("data/tokenized_sentiment_train")

print("Saving tokenized test set...")
tokenized_test.save_to_disk("data/tokenized_sentiment_test")

print("\nTokenized datasets saved successfully!")
print("Train set location: data/tokenized_sentiment_train")
print("Test set location: data/tokenized_sentiment_test")

Saving tokenized training set...


Saving the dataset (0/1 shards):   0%|          | 0/31232 [00:00<?, ? examples/s]

Saving tokenized test set...


Saving the dataset (0/1 shards):   0%|          | 0/5206 [00:00<?, ? examples/s]


Tokenized datasets saved successfully!
Train set location: data/tokenized_sentiment_train
Test set location: data/tokenized_sentiment_test


## 5. Create BoolQ Evaluation Set (from official dataset)

Build a larger BoolQ-style evaluation set from the official Hugging Face dataset `google/boolq` to obtain more stable metrics (default 500 examples).

In [7]:
# Build a larger BoolQ evaluation set from the official dataset
from datasets import load_dataset
import random, json, os

# Reproducible sampling
random.seed(42)

# Load official BoolQ splits
boolq_ds = load_dataset("google/boolq")  # has 'train' and 'validation'

# Helper to format prompt
def make_prompt(passage: str, question: str) -> str:
    return f"Passage: {passage}\nQuestion: {question} (yes or no)\nAnswer:"

# Choose how many evaluation examples to include
NUM_EVAL = 500  # adjust to 100/500/1000 as needed
src_split = boolq_ds["validation"]  # use held-out validation split

indices = list(range(len(src_split)))
random.shuffle(indices)
indices = indices[:min(NUM_EVAL, len(indices))]

boolq_eval = []
for idx in indices:
    ex = src_split[idx]
    boolq_eval.append({
        "question": ex["question"],
        "passage": ex["passage"],
        "answer": "yes" if ex["answer"] else "no",
        "prompt": make_prompt(ex["passage"], ex["question"]),
    })

print(f"Created BoolQ evaluation set with {len(boolq_eval)} examples from google/boolq (validation split)")

# Save BoolQ eval set to disk
os.makedirs("data", exist_ok=True)
with open("data/boolq_eval.json", "w") as f:
    json.dump(boolq_eval, f, indent=2)

print("Saved to: data/boolq_eval.json")

Created BoolQ evaluation set with 500 examples from google/boolq (validation split)
Saved to: data/boolq_eval.json
